# 🚀 Azure SQL Advisor — Recommendation Engine

**Notebook 2 of 2** — Reads telemetry from ADLS Gen2, runs **10 analyzers**,
scores/ranks recommendations, and outputs Parquet + HTML report.

| Analyzer | Category | Sub-checks |
|---|---|---|
| 🗂️ Index | INDEXING | Missing, Unused, Duplicate, Fragmentation, FK, Columnstore, Write-heavy, Density |
| 📋 Stored Procedures | STORED_PROCEDURES | Execution stats, Code inspection (7 anti-patterns), Parameter sniffing |
| 👁️ Views | VIEWS | Nested, SELECT *, SCHEMABINDING, NOLOCK, Indexed view candidates/overhead |
| 📐 Schema | SCHEMA | Heaps, No PK, Wide tables, Data types, LOB columns |
| 💾 Storage | STORAGE | Large tables, Compression, Index ratio, Unused space, File space |
| 📊 Activity | ACTIVITY | Top users, Operation types, Connection patterns |
| 🗄️ Archival | ARCHIVAL | Cold tables, Partition candidates, Data retention |
| 🏎️ Performance | PERFORMANCE | CPU/IO/Memory, Waits, Expensive queries, QS, Blocking, TempDB, Stale stats |
| 💰 Cost | COST | Tier rightsizing, Serverless opportunity, Scale-up/down |
| ⚙️ Operations | OPERATIONS | Query Store, Auto-stats, RCSI, Auto-tuning, Plan cache, Long transactions |

In [ ]:
# Databricks notebook source
# Create interactive widgets
dbutils.widgets.text("server_name", "", "Azure SQL Server FQDN")
dbutils.widgets.text("database_name", "", "Database Name")
dbutils.widgets.text("storage_account", "", "Storage Account Name")
dbutils.widgets.text("storage_container", "azure-sql-telemetry", "Container Name")
dbutils.widgets.dropdown("lookback_days", "7", ["1","3","7","14","30","60","90"], "Lookback Days")

### 📦 Import Configuration

In [ ]:
import sys, os, json
from datetime import datetime, timezone, timedelta

repo_path = os.path.dirname(os.path.abspath(globals().get('__file__', '/Workspace/Repos/azure_sql_advisor')))
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from config import (
    AdvisorConfig, Category, Severity, Effort, Risk, Confidence, Recommendation,
    SEVERITY_SCORES, SEVERITY_SCORES_STR, AZURE_SQL_PRICING,
    WAIT_CATEGORIES, BENIGN_WAIT_TYPES, SP_BEST_PRACTICES,
    CATEGORY_WEIGHT_FIELDS, ANALYSIS_CATEGORIES, OUTPUT_COLUMNS,
    COMPRESSION_ESTIMATES, DTU_TIER_ORDER, VCORE_GP_TIER_ORDER,
    VCORE_BC_TIER_ORDER, VCORE_SERVERLESS_ORDER,
)

# Read widget values
server_name = dbutils.widgets.get("server_name")
database_name = dbutils.widgets.get("database_name")
storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("storage_container")
lookback_days = int(dbutils.widgets.get("lookback_days"))

config = AdvisorConfig(
    server=server_name,
    database=database_name,
    storage_account_name=storage_account,
    storage_container=storage_container,
    lookback_days=lookback_days,
)

base_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/raw/{server_name}/{database_name}"
print(f"Source: {base_path}")
print(f"Lookback: {lookback_days} days")

### 📂 Load Telemetry from ADLS Gen2

Read partitioned Parquet data for each metric.

In [ ]:
from pyspark.sql.functions import col, lit, max as spark_max

def load_metric(metric_name, lookback=lookback_days):
    """Load a metric from ADLS Gen2 Parquet, filtered by lookback window."""
    metric_path = f"{base_path}/{metric_name}"
    try:
        df = spark.read.parquet(metric_path)
        count = df.count()
        print(f"  ✓ {metric_name}: {count} rows")
        return df
    except Exception as e:
        print(f"  ✗ {metric_name}: not available — {str(e)[:100]}")
        return None

# Load all available metrics into a dict
data = {}
metrics_to_load = [
    'resource_stats', 'query_store_stats',
    'database_summary', 'top_queries_cpu', 'top_queries_reads',
    'wait_stats', 'missing_indexes', 'unused_indexes', 'duplicate_indexes',
    'index_fragmentation', 'index_usage_patterns', 'fk_without_index',
    'columnstore_candidates',
    'table_sizes', 'database_files', 'compression_candidates',
    'service_tier', 'data_type_audit',
    'sp_execution_stats', 'sp_source_code', 'sp_parameter_sniffing',
    'views_analysis', 'indexed_view_candidates', 'indexed_view_usage',
    'heap_tables', 'tables_no_pk', 'wide_tables', 'lob_columns',
    'current_activity', 'operation_types', 'connection_patterns',
    'cold_tables', 'partition_candidates',
    'blocking_chains', 'tempdb_usage', 'log_space',
    'stale_statistics', 'plan_cache', 'db_options',
    'query_store_status', 'auto_tuning_recommendations',
    'long_running_transactions',
]

print("Loading telemetry from ADLS Gen2...")
for m in metrics_to_load:
    data[m] = load_metric(m)

# Convert to Python dicts for analyzers
records = {}
for m, df in data.items():
    if df is not None:
        try:
            records[m] = [row.asDict() for row in df.collect()]
        except Exception:
            records[m] = []
    else:
        records[m] = []

print(f"\nLoaded {sum(1 for v in records.values() if v)} metrics with data")

In [ ]:
def make_rec(category, subcategory, severity, object_type, schema_name, object_name,
             recommendation, recommendation_detail, action_sql="",
             estimated_impact_pct=0.0, estimated_savings_mb=0.0,
             effort="MEDIUM", risk="LOW", confidence="HIGH",
             source_dmv="", metric_value="", metric_threshold="",
             estimated_cost_monthly=0.0):
    """Create a single recommendation row as a dict."""
    return {
        "category": category,
        "subcategory": subcategory,
        "severity": severity,
        "object_type": object_type,
        "schema_name": schema_name or "",
        "object_name": object_name or "",
        "recommendation": recommendation,
        "recommendation_detail": recommendation_detail,
        "action_sql": action_sql,
        "estimated_impact_pct": float(estimated_impact_pct) if estimated_impact_pct else 0.0,
        "estimated_savings_mb": float(estimated_savings_mb) if estimated_savings_mb else 0.0,
        "estimated_cost_monthly": float(estimated_cost_monthly) if estimated_cost_monthly else 0.0,
        "effort": effort,
        "risk": risk,
        "confidence": confidence,
        "source_dmv": source_dmv,
        "metric_value": str(metric_value) if metric_value is not None else "",
        "metric_threshold": str(metric_threshold) if metric_threshold is not None else "",
    }

def safe_val(row, key, default=0):
    v = row.get(key)
    return v if v is not None else default

def safe_avg(rows, col_name):
    vals = [float(r[col_name]) for r in rows if r.get(col_name) is not None]
    return sum(vals)/len(vals) if vals else 0.0

def safe_max_val(rows, col_name):
    vals = [float(r[col_name]) for r in rows if r.get(col_name) is not None]
    return max(vals) if vals else 0.0

## 🗂️ Index Analyzer

Analyzes missing, unused, duplicate, fragmented, FK, columnstore, write-heavy indexes, and index density.

In [ ]:
def analyze_indexes(records, config):
    recs = []
    # 1. Missing indexes
    for r in records.get('missing_indexes', []):
        impact = safe_val(r, 'avg_user_impact', 0)
        sev = 'CRITICAL' if impact > 80 else ('HIGH' if impact > 50 else 'MEDIUM')
        key_cols = ', '.join([c for c in [r.get('equality_columns'), r.get('inequality_columns')] if c])
        recs.append(make_rec('INDEXING', 'MISSING_INDEX', sev, 'TABLE',
            '', r.get('table_name', ''),
            'Create missing nonclustered index',
            f'Expected impact {impact}%. Seeks: {safe_val(r,"user_seeks")}, scans: {safe_val(r,"user_scans")}. Key: {key_cols}',
            action_sql=r.get('create_index_ddl', ''),
            estimated_impact_pct=impact, effort='QUICK_WIN', confidence='HIGH',
            source_dmv='sys.dm_db_missing_index_groups', metric_value=impact,
            metric_threshold=config.missing_index_impact_threshold))

    # 2. Unused indexes
    for r in records.get('unused_indexes', []):
        recs.append(make_rec('INDEXING', 'UNUSED_INDEX', 'MEDIUM', 'INDEX',
            r.get('schema_name',''), r.get('index_name',''),
            f'Drop unused index on {r.get("table_name","")}',
            f'Reads: 0, updates: {safe_val(r,"user_updates")}. Size: {safe_val(r,"size_mb",0):.2f} MB',
            action_sql=r.get('drop_index_ddl', ''),
            estimated_impact_pct=5.0, estimated_savings_mb=safe_val(r,'size_mb',0),
            effort='QUICK_WIN', source_dmv='sys.dm_db_index_usage_stats'))

    # 3. Duplicate indexes
    for r in records.get('duplicate_indexes', []):
        recs.append(make_rec('INDEXING', 'DUPLICATE_INDEX', 'MEDIUM', 'INDEX',
            r.get('schema_name',''), r.get('index_b',''),
            f'Drop duplicate index {r.get("index_b","")}',
            f'Duplicate of {r.get("index_a","")}. Key columns: {r.get("key_columns","")}',
            action_sql=f'DROP INDEX [{r.get("index_b","")}] ON [{r.get("schema_name","")}].[{r.get("table_name","")}]',
            estimated_impact_pct=5.0, effort='QUICK_WIN', source_dmv='sys.indexes'))

    # 4. Fragmented indexes
    for r in records.get('index_fragmentation', []):
        frag = safe_val(r, 'avg_fragmentation_in_percent', 0)
        sev = 'HIGH' if frag > 30 else 'MEDIUM'
        recs.append(make_rec('INDEXING', 'FRAGMENTATION', sev, 'INDEX',
            r.get('schema_name',''), r.get('index_name',''),
            'Rebuild or Reorganize Index',
            f'Fragmentation: {frag:.1f}% across {safe_val(r,"page_count")} pages',
            action_sql=r.get('recommended_action', ''),
            estimated_impact_pct=frag/10, source_dmv='sys.dm_db_index_physical_stats',
            metric_value=frag, metric_threshold=config.fragmentation_reorg_pct))

    # 5. FK without supporting index
    for r in records.get('fk_without_index', []):
        recs.append(make_rec('INDEXING', 'FOREIGN_KEY_INDEX', 'MEDIUM', 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Create supporting index for foreign key',
            f'FK {r.get("fk_name","")} on column {r.get("column_name","")} has {safe_val(r,"row_count"):,} child rows',
            action_sql=f'CREATE NONCLUSTERED INDEX [IX_{r.get("table_name","")}_{r.get("column_name","")}] ON [{r.get("schema_name","")}].[{r.get("table_name","")}] ([{r.get("column_name","")}])',
            estimated_impact_pct=12.0, effort='LOW', source_dmv='sys.foreign_keys'))

    # 6. Columnstore candidates
    for r in records.get('columnstore_candidates', []):
        scans = safe_val(r, 'scan_count', 0)
        seeks = safe_val(r, 'seek_count', 1)
        if scans > seeks * 2:
            recs.append(make_rec('INDEXING', 'COLUMNSTORE_CANDIDATE', 'MEDIUM', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Consider Columnstore Index for analytical workload',
                f'Table has {safe_val(r,"row_count"):,} rows ({safe_val(r,"size_mb"):.0f} MB). Scans: {scans}, seeks: {seeks}',
                estimated_impact_pct=30.0, estimated_savings_mb=safe_val(r,'size_mb',0)*0.9,
                effort='HIGH', risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_db_partition_stats'))

    # 7. Write-heavy indexes
    for r in records.get('index_usage_patterns', []):
        reads = safe_val(r,'user_seeks',0)+safe_val(r,'user_scans',0)+safe_val(r,'user_lookups',0)
        updates = safe_val(r,'user_updates',0)
        if updates > config.write_heavy_index_min_updates and updates > reads*10:
            recs.append(make_rec('INDEXING', 'WRITE_HEAVY_INDEX', 'LOW', 'INDEX',
                r.get('schema_name',''), r.get('index_name',''),
                'Review write-heavy index',
                f'Reads: {reads:,}, updates: {updates:,}. Pattern: {r.get("usage_pattern","")}',
                effort='MEDIUM', risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_db_index_usage_stats'))

    return recs

index_recs = analyze_indexes(records, config)
print(f'Index Recommendations: {len(index_recs)}')

## 📋 Stored Procedure Analyzer

Analyzes SP execution stats (CPU/IO/Duration/Writes/Recompile), code inspection (7 anti-patterns), and parameter sniffing.

In [ ]:
def analyze_stored_procedures(records, config):
    recs = []
    # 1. Execution stats
    for r in records.get('sp_execution_stats', []):
        avg_cpu = safe_val(r, 'avg_cpu_ms', 0)
        avg_reads = safe_val(r, 'avg_logical_reads', 0)
        avg_duration = safe_val(r, 'avg_duration_ms', 0)
        avg_writes = safe_val(r, 'avg_logical_writes', 0)
        recompiles = safe_val(r, 'recompile_count', 0)

        if avg_cpu > config.sp_high_cpu_ms:
            recs.append(make_rec('STORED_PROCEDURES', 'HIGH_CPU', 'CRITICAL', 'PROCEDURE',
                r.get('schema_name',''), r.get('procedure_name',''),
                'Review execution plan for CPU bottlenecks',
                f'Avg CPU: {avg_cpu:.1f}ms, executions: {safe_val(r,"execution_count")}',
                estimated_impact_pct=20.0, source_dmv='sys.dm_exec_procedure_stats',
                metric_value=avg_cpu, metric_threshold=config.sp_high_cpu_ms))

        if avg_reads > config.sp_high_reads:
            recs.append(make_rec('STORED_PROCEDURES', 'HIGH_IO', 'HIGH', 'PROCEDURE',
                r.get('schema_name',''), r.get('procedure_name',''),
                'Review indexing or query shape for logical reads',
                f'Avg reads: {avg_reads:,} per execution',
                estimated_impact_pct=15.0, source_dmv='sys.dm_exec_procedure_stats',
                metric_value=avg_reads, metric_threshold=config.sp_high_reads))

        if avg_duration > config.sp_high_duration_ms:
            recs.append(make_rec('STORED_PROCEDURES', 'HIGH_DURATION', 'HIGH', 'PROCEDURE',
                r.get('schema_name',''), r.get('procedure_name',''),
                'Tune long-running stored procedure',
                f'Avg duration: {avg_duration:.1f}ms, avg CPU: {avg_cpu:.1f}ms, avg reads: {avg_reads:,}',
                estimated_impact_pct=20.0, source_dmv='sys.dm_exec_procedure_stats',
                metric_value=avg_duration, metric_threshold=config.sp_high_duration_ms))

        if avg_writes > config.sp_high_writes:
            recs.append(make_rec('STORED_PROCEDURES', 'HIGH_WRITES', 'MEDIUM', 'PROCEDURE',
                r.get('schema_name',''), r.get('procedure_name',''),
                'Review write-heavy stored procedure',
                f'Avg logical writes: {avg_writes:,} per execution',
                estimated_impact_pct=10.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_exec_procedure_stats'))

        if recompiles > config.sp_recompile_threshold:
            recs.append(make_rec('STORED_PROCEDURES', 'HIGH_RECOMPILE', 'HIGH', 'PROCEDURE',
                r.get('schema_name',''), r.get('procedure_name',''),
                'Investigate high recompile count — consider OPTIMIZE FOR',
                f'Recompiles: {recompiles}',
                estimated_impact_pct=10.0, effort='HIGH', risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_exec_procedure_stats'))

    # 2. Code inspection (7 anti-patterns)
    for r in records.get('sp_source_code', []):
        sch = r.get('schema_name', '')
        sp = r.get('procedure_name', '')

        if safe_val(r, 'missing_nocount') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'BEST_PRACTICE', 'LOW', 'PROCEDURE',
                sch, sp, 'Add SET NOCOUNT ON', 'Reduces network traffic',
                effort='QUICK_WIN', source_dmv='sys.sql_modules'))
        if safe_val(r, 'has_select_star') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'BEST_PRACTICE', 'LOW', 'PROCEDURE',
                sch, sp, 'Avoid SELECT *', 'Explicitly list columns for better plan quality',
                effort='LOW', source_dmv='sys.sql_modules'))
        if safe_val(r, 'uses_cursor') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'BEST_PRACTICE', 'MEDIUM', 'PROCEDURE',
                sch, sp, 'Replace CURSOR with set-based logic', 'Cursors are orders of magnitude slower',
                estimated_impact_pct=10.0, effort='HIGH', risk='MEDIUM',
                source_dmv='sys.sql_modules'))
        if safe_val(r, 'uses_dynamic_exec') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'BEST_PRACTICE', 'MEDIUM', 'PROCEDURE',
                sch, sp, 'Use sp_executesql instead of EXEC()', 'Prevents SQL injection, allows plan reuse',
                estimated_impact_pct=5.0, risk='MEDIUM', source_dmv='sys.sql_modules'))
        if safe_val(r, 'uses_table_variable') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'CARDINALITY_ESTIMATE', 'MEDIUM', 'PROCEDURE',
                sch, sp, 'Review table variable usage',
                'Table variables produce 1-row cardinality estimates. Use #temp tables for larger results.',
                estimated_impact_pct=8.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.sql_modules'))
        if safe_val(r, 'uses_option_recompile') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'RECOMPILE_HINT', 'LOW', 'PROCEDURE',
                sch, sp, 'Validate OPTION(RECOMPILE) cost/benefit',
                'Keep only where runtime savings exceed compile overhead',
                effort='LOW', confidence='MEDIUM', source_dmv='sys.sql_modules'))
        if safe_val(r, 'transaction_without_try') == 1:
            recs.append(make_rec('STORED_PROCEDURES', 'TRANSACTION_SAFETY', 'MEDIUM', 'PROCEDURE',
                sch, sp, 'Wrap explicit transactions in TRY/CATCH',
                'Open transactions after errors cause blocking and log growth',
                estimated_impact_pct=8.0, confidence='MEDIUM', source_dmv='sys.sql_modules'))

    # 3. Parameter sniffing
    for r in records.get('sp_parameter_sniffing', []):
        risk_level = r.get('sniffing_risk', 'LOW')
        sev = 'HIGH' if risk_level == 'SEVERE' else 'MEDIUM'
        recs.append(make_rec('STORED_PROCEDURES', 'PARAMETER_SNIFFING', sev, 'PROCEDURE',
            r.get('schema_name',''), r.get('procedure_name',''),
            'Parameter sniffing detected',
            f'CPU variance: min={safe_val(r,"min_cpu_ms")}ms, max={safe_val(r,"max_cpu_ms")}ms, avg={safe_val(r,"avg_cpu_ms")}ms',
            action_sql='-- ALTER PROCEDURE ... WITH RECOMPILE  OR  Use local variables',
            estimated_impact_pct=15.0, confidence='MEDIUM',
            source_dmv='sys.dm_exec_procedure_stats'))

    return recs

sp_recs = analyze_stored_procedures(records, config)
print(f'SP Recommendations: {len(sp_recs)}')

## 👁️ Views Analyzer

Analyzes nested views, SELECT *, SCHEMABINDING, NOLOCK hints, indexed view candidates and overhead.

In [ ]:
def analyze_views(records, config):
    recs = []
    for r in records.get('views_analysis', []):
        sch = r.get('schema_name', '')
        vn = r.get('view_name', '')

        nested = safe_val(r, 'nested_view_count', 0)
        if nested > config.view_nested_threshold:
            recs.append(make_rec('VIEWS', 'NESTED_VIEW', 'MEDIUM', 'VIEW',
                sch, vn, 'Flatten deeply nested view',
                f'View references {nested} other views. Deep nesting creates complex plans.',
                estimated_impact_pct=5.0, effort='HIGH', risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.views', metric_value=nested, metric_threshold=config.view_nested_threshold))

        if safe_val(r, 'has_select_star') == 1:
            recs.append(make_rec('VIEWS', 'SELECT_STAR', 'LOW', 'VIEW',
                sch, vn, 'Replace SELECT * with explicit column list in view',
                'Can cause schema binding issues and transfer unnecessary data.',
                effort='QUICK_WIN', source_dmv='sys.views'))

        if safe_val(r, 'missing_schemabinding') == 1:
            recs.append(make_rec('VIEWS', 'MISSING_SCHEMABINDING', 'LOW', 'VIEW',
                sch, vn, 'Consider WITH SCHEMABINDING for view stability',
                'Prevents accidental schema changes that could break the view.',
                action_sql=f'ALTER VIEW [{sch}].[{vn}] WITH SCHEMABINDING AS ...',
                effort='MEDIUM', risk='MEDIUM', source_dmv='sys.views'))

        length = safe_val(r, 'definition_length', 0)
        if length > config.view_complex_length:
            recs.append(make_rec('VIEWS', 'COMPLEX_VIEW', 'LOW', 'VIEW',
                sch, vn, 'Review complex view for simplification',
                f'View definition is {length} characters. Consider breaking into smaller views.',
                effort='HIGH', confidence='LOW', source_dmv='sys.views',
                metric_value=length, metric_threshold=config.view_complex_length))

        if safe_val(r, 'uses_nolock') == 1:
            recs.append(make_rec('VIEWS', 'NOLOCK_HINT', 'MEDIUM', 'VIEW',
                sch, vn, 'Review NOLOCK usage in view',
                'NOLOCK can return dirty/phantom reads. Use READ COMMITTED SNAPSHOT instead.',
                effort='MEDIUM', source_dmv='sys.views'))

    # Indexed view candidates
    for r in records.get('indexed_view_candidates', []):
        reads = safe_val(r, 'total_reads', 0)
        if reads > config.indexed_view_reads_threshold:
            recs.append(make_rec('VIEWS', 'INDEXED_VIEW_CANDIDATE', 'MEDIUM', 'VIEW',
                r.get('schema_name',''), r.get('view_name',''),
                'Consider creating an indexed (materialized) view',
                f'View has {reads} reads. Add SCHEMABINDING, then CREATE UNIQUE CLUSTERED INDEX.',
                estimated_impact_pct=20.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.views', metric_value=reads, metric_threshold=config.indexed_view_reads_threshold))

    # Indexed view overhead
    for r in records.get('indexed_view_usage', []):
        reads = safe_val(r, 'total_reads', 0)
        writes = safe_val(r, 'total_writes', 0)
        if writes > 10000 and writes > reads * config.indexed_view_write_ratio:
            recs.append(make_rec('VIEWS', 'INDEXED_VIEW_OVERHEAD', 'MEDIUM', 'VIEW',
                r.get('schema_name',''), r.get('view_name',''),
                'Review indexed view maintenance overhead',
                f'Writes ({writes:,}) >> Reads ({reads:,}). Drop nonessential indexed view indexes.',
                estimated_impact_pct=8.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.views + sys.dm_db_index_usage_stats'))

    return recs

view_recs = analyze_views(records, config)
print(f'Views Recommendations: {len(view_recs)}')

## 📐 Schema Analyzer

Analyzes heap tables, missing PKs, wide tables, data type audit, and LOB columns.

In [ ]:
def analyze_schema(records, config):
    recs = []
    # 1. Heap tables
    for r in records.get('heap_tables', []):
        size = safe_val(r, 'size_mb', 0)
        if size > 10:
            recs.append(make_rec('SCHEMA', 'HEAP_TABLE', 'HIGH', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Create Clustered Index', f'Heap table: {size:.1f} MB, {safe_val(r,"row_count"):,} rows',
                estimated_impact_pct=15.0, effort='MEDIUM', source_dmv='sys.tables'))

    # 2. Tables without PK
    for r in records.get('tables_no_pk', []):
        recs.append(make_rec('SCHEMA', 'NO_PRIMARY_KEY', 'MEDIUM', 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Define a Primary Key', f'Rows: {safe_val(r,"row_count"):,}',
            estimated_impact_pct=5.0, effort='MEDIUM', risk='MEDIUM', source_dmv='sys.tables'))

    # 3. Wide tables
    for r in records.get('wide_tables', []):
        recs.append(make_rec('SCHEMA', 'WIDE_TABLE', 'LOW', 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Review table design for normalization',
            f'Columns: {safe_val(r,"column_count")}',
            effort='HIGH', risk='HIGH', confidence='MEDIUM',
            source_dmv='INFORMATION_SCHEMA.COLUMNS',
            metric_value=safe_val(r,'column_count'), metric_threshold=config.wide_table_column_threshold))

    # 4. Data type audit (NVARCHAR)
    nvarchar_counts = {}
    for r in records.get('data_type_audit', []):
        dt = r.get('data_type', '')
        if dt.lower() in ('nvarchar', 'nchar', 'ntext'):
            tbl = f"{r.get('schema_name','')}.{r.get('table_name','')}"
            if tbl not in nvarchar_counts:
                nvarchar_counts[tbl] = {'schema': r.get('schema_name',''), 'table': r.get('table_name',''), 'count': 0}
            nvarchar_counts[tbl]['count'] += 1
    for v in nvarchar_counts.values():
        if v['count'] > config.nvarchar_column_alert_count:
            recs.append(make_rec('SCHEMA', 'DATA_TYPE', 'LOW', 'TABLE',
                v['schema'], v['table'],
                'Review NVARCHAR usage', f'Has {v["count"]} NVARCHAR columns. Switch to VARCHAR if Unicode not needed.',
                estimated_impact_pct=5.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='INFORMATION_SCHEMA.COLUMNS'))

    # 5. LOB columns
    for r in records.get('lob_columns', []):
        recs.append(make_rec('SCHEMA', 'LOB_COLUMNS', 'LOW', 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Review LOB/MAX column storage',
            f'LOB/MAX columns: {r.get("lob_columns","")}. Consider externalizing cold large payloads.',
            estimated_impact_pct=5.0, effort='HIGH', risk='MEDIUM', confidence='MEDIUM',
            source_dmv='INFORMATION_SCHEMA.COLUMNS'))

    return recs

schema_recs = analyze_schema(records, config)
print(f'Schema Recommendations: {len(schema_recs)}')

## 💾 Storage Analyzer

Audits large tables, compression opportunities, index/data ratio, unused space, and database files.

In [ ]:
def analyze_storage(records, config):
    recs = []
    for r in records.get('table_sizes', []):
        reserved = safe_val(r, 'reserved_mb', 0)
        data_mb = safe_val(r, 'data_mb', 0)
        index_mb = safe_val(r, 'index_mb', 0)
        unused_mb = safe_val(r, 'unused_mb', 0)
        unused_pct = (unused_mb/reserved)*100 if reserved > 0 else 0
        idx_ratio = (index_mb/data_mb) if data_mb > 0 else 0

        if reserved > config.table_size_concern_gb * 1024:
            recs.append(make_rec('STORAGE', 'LARGE_TABLE', 'MEDIUM', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Review large table storage strategy',
                f'Reserved: {reserved:.1f} MB, data: {data_mb:.1f} MB, rows: {safe_val(r,"row_count"):,}',
                estimated_impact_pct=10.0, source_dmv='sys.dm_db_partition_stats'))

        if idx_ratio > config.high_index_to_data_ratio and index_mb > 512:
            recs.append(make_rec('STORAGE', 'INDEX_STORAGE_RATIO', 'MEDIUM', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Reduce index storage overhead',
                f'Index/data ratio: {idx_ratio:.2f}x ({index_mb:.1f} MB indexes vs {data_mb:.1f} MB data)',
                estimated_impact_pct=8.0, estimated_savings_mb=index_mb*0.2,
                risk='MEDIUM', confidence='MEDIUM', source_dmv='sys.dm_db_partition_stats'))

        if unused_mb > config.unused_space_min_mb and unused_pct > config.unused_space_pct_threshold:
            recs.append(make_rec('STORAGE', 'UNUSED_RESERVED_SPACE', 'LOW', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Investigate unused reserved space',
                f'Unused: {unused_mb:.1f} MB ({unused_pct:.1f}% of reserved)',
                estimated_savings_mb=unused_mb, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_db_partition_stats'))

    # Compression candidates
    for r in records.get('compression_candidates', []):
        size = safe_val(r, 'size_mb', 0)
        if size > 100:
            recs.append(make_rec('STORAGE', 'COMPRESSION', 'MEDIUM', 'TABLE',
                r.get('schema_name',''), r.get('table_name',''),
                'Apply PAGE compression',
                f'Table size: {size:.1f} MB, uncompressed. Estimated savings: ~{size*0.4:.0f} MB',
                action_sql=f'ALTER TABLE [{r.get("schema_name","")}].[{r.get("table_name","")}] REBUILD WITH (DATA_COMPRESSION = PAGE)',
                estimated_impact_pct=10.0, estimated_savings_mb=size*0.4,
                effort='LOW', source_dmv='sys.partitions'))

    # Database files
    for r in records.get('database_files', []):
        free = safe_val(r, 'free_mb', 0)
        used_pct = safe_val(r, 'used_pct', 100)
        if free > 1024 and used_pct < 50:
            recs.append(make_rec('STORAGE', 'FILE_SPACE', 'LOW', 'FILE',
                '', r.get('file_name',''),
                'Consider shrinking database file',
                f'File is {100-used_pct:.1f}% empty ({free:.1f} MB free). Can cause fragmentation.',
                estimated_savings_mb=free, risk='MEDIUM', source_dmv='sys.database_files'))

    return recs

storage_recs = analyze_storage(records, config)
print(f'Storage Recommendations: {len(storage_recs)}')

## 📊 Activity Analyzer

Analyzes top resource consumers, operation type breakdown, and connection patterns.

In [ ]:
def analyze_activity(records, config):
    recs = []
    # Top users
    for r in records.get('current_activity', []):
        cpu = safe_val(r, 'session_cpu_time', 0)
        if cpu > 100000:
            recs.append(make_rec('ACTIVITY', 'TOP_USER', 'INFO', 'USER',
                '', r.get('login_name', 'unknown'),
                f'High resource user from {r.get("host_name","unknown")}',
                f'CPU: {cpu/1000:.1f}s, Reads: {safe_val(r,"session_reads"):,}, Program: {r.get("program_name","unknown")}',
                source_dmv='sys.dm_exec_sessions'))

    # DDL operations
    for r in records.get('operation_types', []):
        if r.get('operation_type') == 'DDL' and safe_val(r, 'total_executions', 0) > 100:
            recs.append(make_rec('ACTIVITY', 'DDL_OPERATIONS', 'INFO', 'DATABASE',
                '', config.database,
                'Review frequent DDL operations',
                f'Frequent DDL can cause schema locks. Executions: {safe_val(r,"total_executions")}',
                source_dmv='sys.dm_exec_query_stats'))

    # Connection patterns
    for r in records.get('connection_patterns', []):
        conns = safe_val(r, 'connection_count', 0)
        idle = safe_val(r, 'idle_connections', 0)
        if conns > 50 and idle > conns * 0.8:
            recs.append(make_rec('ACTIVITY', 'IDLE_CONNECTIONS', 'MEDIUM', 'APP',
                '', r.get('program_name', ''),
                'Implement connection pooling',
                f'{idle} idle out of {conns} connections from {r.get("host_name","")}',
                estimated_impact_pct=10.0, source_dmv='sys.dm_exec_sessions'))

    return recs

activity_recs = analyze_activity(records, config)
print(f'Activity Recommendations: {len(activity_recs)}')

## 🗄️ Archival Analyzer

Identifies cold data for archiving, partition candidates, and data retention opportunities.

In [ ]:
def analyze_archival(records, config):
    recs = []
    # Cold tables
    for r in records.get('cold_tables', []):
        days = safe_val(r, 'days_since_last_read', 0)
        size = safe_val(r, 'size_mb', 0)
        if days > config.cold_table_critical_days and size > 1024:
            sev = 'CRITICAL'
        elif days > config.cold_table_high_days and size > 100:
            sev = 'HIGH'
        elif days > config.cold_table_medium_days:
            sev = 'MEDIUM'
        else:
            continue
        recs.append(make_rec('ARCHIVAL', 'COLD_DATA', sev, 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Archive cold data to Azure Blob Storage',
            f'Table idle for {days} days. Size: {size:.1f} MB',
            estimated_impact_pct=20.0, estimated_savings_mb=size,
            effort='HIGH', risk='MEDIUM', source_dmv='sys.dm_db_index_usage_stats',
            metric_value=days, metric_threshold=config.cold_table_medium_days))

    # Partition candidates
    for r in records.get('partition_candidates', []):
        recs.append(make_rec('ARCHIVAL', 'PARTITIONING', 'MEDIUM', 'TABLE',
            r.get('schema_name',''), r.get('table_name',''),
            'Implement RANGE RIGHT partitioning',
            f'Large table ({safe_val(r,"size_mb"):.1f} MB) with datetime column {r.get("datetime_column","")}',
            estimated_impact_pct=15.0, effort='HIGH', risk='MEDIUM', confidence='MEDIUM',
            source_dmv='INFORMATION_SCHEMA.COLUMNS'))

    return recs

archival_recs = analyze_archival(records, config)
print(f'Archival Recommendations: {len(archival_recs)}')

## 🏎️ Performance Analyzer

CPU/IO/Memory pressure, wait statistics, expensive queries, Query Store, blocking, TempDB, stale stats.

In [ ]:
def analyze_performance(records, config):
    recs = []
    res = records.get('resource_stats', [])

    if res:
        avg_cpu = safe_avg(res, 'avg_cpu_percent')
        avg_io = safe_avg(res, 'avg_data_io_percent')
        avg_log = safe_avg(res, 'avg_log_write_percent')
        avg_mem = safe_avg(res, 'avg_memory_usage_percent')
        max_worker = safe_max_val(res, 'max_worker_percent')
        max_session = safe_max_val(res, 'max_session_percent')

        if avg_cpu > 80: sev = 'CRITICAL'
        elif avg_cpu > 60: sev = 'HIGH'
        elif avg_cpu > 40: sev = 'MEDIUM'
        else: sev = None
        if sev:
            recs.append(make_rec('PERFORMANCE', 'CPU_UTILIZATION', sev, 'DATABASE',
                '', config.database, 'Review CPU usage, consider scaling up',
                f'Average CPU: {avg_cpu:.1f}%', estimated_impact_pct=25.0,
                source_dmv='sys.dm_db_resource_stats', metric_value=f'{avg_cpu:.1f}%', metric_threshold='80%'))

        if avg_io > config.io_high_pct:
            sev = 'CRITICAL' if avg_io > config.io_critical_pct else 'HIGH'
            recs.append(make_rec('PERFORMANCE', 'DATA_IO_PRESSURE', sev, 'DATABASE',
                '', config.database, 'Reduce data IO pressure',
                f'Average data IO: {avg_io:.1f}%. Prioritize missing indexes, page compression.',
                estimated_impact_pct=20.0, source_dmv='sys.dm_db_resource_stats'))

        if avg_log > config.io_high_pct:
            sev = 'CRITICAL' if avg_log > config.io_critical_pct else 'HIGH'
            recs.append(make_rec('PERFORMANCE', 'LOG_WRITE_PRESSURE', sev, 'DATABASE',
                '', config.database, 'Reduce transaction log write pressure',
                f'Average log write: {avg_log:.1f}%. Batch writes, reduce redundant indexes.',
                estimated_impact_pct=20.0, source_dmv='sys.dm_db_resource_stats'))

        if avg_mem > config.memory_high_pct:
            recs.append(make_rec('PERFORMANCE', 'MEMORY_PRESSURE', 'HIGH', 'DATABASE',
                '', config.database, 'Investigate memory pressure',
                f'Average memory: {avg_mem:.1f}%', estimated_impact_pct=15.0,
                source_dmv='sys.dm_db_resource_stats'))

        if max_worker > 80 or max_session > 80:
            recs.append(make_rec('PERFORMANCE', 'CONCURRENCY_PRESSURE', 'HIGH', 'DATABASE',
                '', config.database, 'Investigate worker/session pressure',
                f'Peak workers: {max_worker:.1f}%, sessions: {max_session:.1f}%',
                estimated_impact_pct=15.0, source_dmv='sys.dm_db_resource_stats'))

    # Wait stats
    waits = records.get('wait_stats', [])
    if waits:
        wait_cats = {}
        for r in waits:
            wt = r.get('wait_type', '')
            pct = safe_val(r, 'pct_of_total', 0)
            cat = WAIT_CATEGORIES.get(wt, 'OTHER')
            if cat not in wait_cats:
                wait_cats[cat] = {'total_pct': 0, 'waits': []}
            wait_cats[cat]['total_pct'] += pct
            wait_cats[cat]['waits'].append(wt)
        for cat, info in sorted(wait_cats.items(), key=lambda x: x[1]['total_pct'], reverse=True)[:3]:
            if info['total_pct'] > 20:
                sev = 'HIGH' if info['total_pct'] > 40 else 'MEDIUM'
                recs.append(make_rec('PERFORMANCE', f'WAIT_{cat}', sev, 'DATABASE',
                    '', config.database,
                    f'Dominant wait category: {cat} ({info["total_pct"]:.1f}%)',
                    f'Top waits: {", ".join(info["waits"][:3])}',
                    estimated_impact_pct=info['total_pct']*0.3,
                    source_dmv='sys.dm_os_wait_stats'))

    # Expensive queries
    for r in records.get('top_queries_cpu', []):
        avg_cpu_q = safe_val(r, 'avg_cpu_ms', 0)
        avg_dur_q = safe_val(r, 'avg_duration_ms', 0)
        avg_reads_q = safe_val(r, 'avg_logical_reads', 0)
        if avg_dur_q > config.query_duration_high_ms:
            recs.append(make_rec('PERFORMANCE', 'LONG_RUNNING_QUERY', 'HIGH', 'QUERY',
                '', f'Hash: {r.get("query_hash","")}',
                'Tune long-running query',
                f'Avg CPU: {avg_cpu_q:.1f}ms, duration: {avg_dur_q:.1f}ms, reads: {avg_reads_q:,}. Text: {str(r.get("query_text",""))[:100]}',
                estimated_impact_pct=15.0, effort='HIGH', risk='MEDIUM',
                source_dmv='sys.dm_exec_query_stats'))
        elif avg_cpu_q > config.query_duration_low_ms:
            recs.append(make_rec('PERFORMANCE', 'EXPENSIVE_QUERY', 'MEDIUM', 'QUERY',
                '', f'Hash: {r.get("query_hash","")}',
                'Tune CPU-heavy query',
                f'Avg CPU: {avg_cpu_q:.1f}ms, reads: {avg_reads_q:,}. Text: {str(r.get("query_text",""))[:100]}',
                estimated_impact_pct=10.0, effort='HIGH',
                source_dmv='sys.dm_exec_query_stats'))

    # Blocking chains
    for r in records.get('blocking_chains', []):
        wait_sec = safe_val(r, 'wait_time_seconds', 0)
        sev = 'CRITICAL' if wait_sec > 60 else ('HIGH' if wait_sec > 10 else 'MEDIUM')
        recs.append(make_rec('PERFORMANCE', 'BLOCKING', sev, 'SESSION',
            '', f'Session {safe_val(r,"blocked_session_id")} blocked by {safe_val(r,"blocking_session_id")}',
            'Active blocking detected',
            f'Wait: {wait_sec}s, Type: {r.get("wait_type","")}, Blocker: {r.get("blocking_login","")}',
            estimated_impact_pct=30.0, effort='QUICK_WIN', risk='HIGH',
            source_dmv='sys.dm_exec_requests'))

    # TempDB
    for r in records.get('tempdb_usage', []):
        total = safe_val(r, 'total_tempdb_mb', 0)
        if total > 100:
            sev = 'HIGH' if total > 500 else 'MEDIUM'
            recs.append(make_rec('PERFORMANCE', 'TEMPDB_USAGE', sev, 'SESSION',
                '', f'Session {safe_val(r,"session_id")} ({r.get("login_name","")})',
                'High TempDB usage detected',
                f'TempDB: {total:.1f} MB. Program: {r.get("program_name","")}',
                estimated_impact_pct=15.0, source_dmv='sys.dm_db_session_space_usage'))

    # Stale statistics
    for r in records.get('stale_statistics', []):
        days = safe_val(r, 'days_since_update', 0)
        mods = safe_val(r, 'modification_counter', 0)
        total_rows = safe_val(r, 'table_rows', 1)
        mod_pct = (mods/total_rows)*100 if total_rows > 0 else 0
        if mod_pct > 20 or days > 30:
            sev = 'HIGH' if mod_pct > 50 or days > 60 else 'MEDIUM'
            recs.append(make_rec('PERFORMANCE', 'STALE_STATISTICS', sev, 'STATISTICS',
                r.get('schema_name',''), f'{r.get("table_name","")}.{r.get("stats_name","")}',
                'Update stale statistics',
                f'Last updated {days} days ago. {mods:,} modifications ({mod_pct:.1f}% of {total_rows:,} rows).',
                action_sql=f'UPDATE STATISTICS [{r.get("schema_name","")}].[{r.get("table_name","")}] [{r.get("stats_name","")}] WITH FULLSCAN',
                estimated_impact_pct=15.0, effort='QUICK_WIN',
                source_dmv='sys.dm_db_stats_properties'))

    # Long running transactions
    for r in records.get('long_running_transactions', []):
        dur = safe_val(r, 'duration_seconds', 0)
        sev = 'CRITICAL' if dur > 3600 else ('HIGH' if dur > 600 else 'MEDIUM')
        recs.append(make_rec('PERFORMANCE', 'LONG_TRANSACTION', sev, 'SESSION',
            '', f'Session {safe_val(r,"session_id")} ({r.get("login_name","")})',
            'Long-running open transaction',
            f'Duration: {dur}s, Log used: {safe_val(r,"log_used_mb"):.1f} MB. Program: {r.get("program_name","")}',
            estimated_impact_pct=25.0, effort='QUICK_WIN', risk='HIGH',
            source_dmv='sys.dm_tran_active_transactions'))

    return recs

perf_recs = analyze_performance(records, config)
print(f'Performance Recommendations: {len(perf_recs)}')

## 💰 Cost Analyzer

Evaluates service tier rightsizing, serverless opportunities, and scale-up/down recommendations.

In [ ]:
def analyze_cost(records, config):
    recs = []
    res = records.get('resource_stats', [])
    tier_data = records.get('service_tier', [])

    if not res or not tier_data:
        return recs

    avg_cpu = safe_avg(res, 'avg_cpu_percent')
    avg_io = safe_avg(res, 'avg_data_io_percent')
    tier = tier_data[0]
    current_slo = str(tier.get('service_objective', '')).strip()
    edition = str(tier.get('edition', '')).strip()

    current_pricing = AZURE_SQL_PRICING.get(current_slo)

    # Underutilized → scale down
    if avg_cpu < config.underutilized_cpu_pct and avg_io < config.underutilized_io_pct:
        recs.append(make_rec('COST', 'UNDERUTILIZED', 'MEDIUM', 'DATABASE',
            '', config.database,
            'Consider scaling down service tier',
            f'Avg CPU: {avg_cpu:.1f}%, data IO: {avg_io:.1f}%. Resources are underutilized.',
            estimated_impact_pct=30.0, effort='LOW', risk='MEDIUM', confidence='MEDIUM',
            source_dmv='sys.dm_db_resource_stats'))

        # Serverless opportunity
        if 'GeneralPurpose' in edition.replace(' ', '') and 'Serverless' not in (current_pricing or {}).get('name', ''):
            recs.append(make_rec('COST', 'SERVERLESS_OPPORTUNITY', 'MEDIUM', 'DATABASE',
                '', config.database,
                'Consider vCore Serverless tier',
                f'Low avg CPU ({avg_cpu:.1f}%) suggests intermittent workload — serverless auto-pause can save significantly.',
                effort='LOW', risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.dm_db_resource_stats'))

    # Overutilized → scale up
    if avg_cpu > config.cpu_high_pct or avg_io > config.io_high_pct:
        recs.append(make_rec('COST', 'CAPACITY_RIGHTSIZING', 'HIGH', 'DATABASE',
            '', config.database,
            f'Evaluate scale-up from {current_slo}',
            f'CPU: {avg_cpu:.1f}%, IO: {avg_io:.1f}%. Tune first, then scale if pressure persists.',
            effort='LOW', risk='MEDIUM', confidence='MEDIUM',
            source_dmv='sys.dm_db_resource_stats'))

    return recs

cost_recs = analyze_cost(records, config)
print(f'Cost Recommendations: {len(cost_recs)}')

## ⚙️ Operations Analyzer

Query Store status, auto-stats, RCSI, auto-tuning, plan cache, log space.

In [ ]:
def analyze_operations(records, config):
    recs = []

    # Query Store status
    qs_data = records.get('query_store_status', [])
    qs_enabled = False
    if qs_data:
        qs_state = str(qs_data[0].get('actual_state_desc', '')).strip()
        qs_enabled = qs_state in ('READ_WRITE', 'READ_ONLY')
    if not qs_enabled:
        recs.append(make_rec('OPERATIONS', 'QUERY_STORE', 'MEDIUM', 'DATABASE',
            '', config.database,
            'Enable Query Store for better diagnostics',
            'Query Store provides historical performance data, regression detection, and plan forcing.',
            action_sql=f'ALTER DATABASE [{config.database}] SET QUERY_STORE = ON (OPERATION_MODE = READ_WRITE)',
            effort='QUICK_WIN', source_dmv='sys.database_query_store_options'))

    # Database options
    for r in records.get('db_options', []):
        if safe_val(r, 'is_auto_create_stats_on') == 0:
            recs.append(make_rec('OPERATIONS', 'AUTO_CREATE_STATS', 'HIGH', 'DATABASE',
                '', config.database, 'Enable auto-create statistics',
                'Auto-create statistics is OFF. The optimizer needs column statistics for cardinality estimates.',
                action_sql=f'ALTER DATABASE [{config.database}] SET AUTO_CREATE_STATISTICS ON',
                estimated_impact_pct=15.0, effort='QUICK_WIN',
                source_dmv='sys.databases'))
        if safe_val(r, 'is_auto_update_stats_on') == 0:
            recs.append(make_rec('OPERATIONS', 'AUTO_UPDATE_STATS', 'HIGH', 'DATABASE',
                '', config.database, 'Enable auto-update statistics',
                'Auto-update statistics is OFF. Plans cannot adapt as data distribution changes.',
                action_sql=f'ALTER DATABASE [{config.database}] SET AUTO_UPDATE_STATISTICS ON',
                estimated_impact_pct=15.0, effort='QUICK_WIN',
                source_dmv='sys.databases'))
        if safe_val(r, 'is_read_committed_snapshot_on') == 0:
            recs.append(make_rec('OPERATIONS', 'READ_COMMITTED_SNAPSHOT', 'MEDIUM', 'DATABASE',
                '', config.database, 'Consider enabling READ_COMMITTED_SNAPSHOT',
                'RCSI reduces reader/writer blocking for OLTP workloads.',
                action_sql=f'ALTER DATABASE [{config.database}] SET READ_COMMITTED_SNAPSHOT ON WITH ROLLBACK IMMEDIATE',
                estimated_impact_pct=10.0, risk='MEDIUM', confidence='MEDIUM',
                source_dmv='sys.databases'))

    # Auto-tuning recommendations
    for r in records.get('auto_tuning_recommendations', []):
        recs.append(make_rec('OPERATIONS', 'AUTO_TUNING', 'MEDIUM', 'DATABASE',
            '', r.get('recommendation_name', config.database),
            f'Azure Auto-Tuning: {r.get("recommendation_type","")}',
            f'Reason: {r.get("reason","")}. Expected improvement: {r.get("expected_improvement","N/A")}',
            action_sql=r.get('implementation_script', ''),
            estimated_impact_pct=10.0, effort='QUICK_WIN',
            source_dmv='sys.dm_db_tuning_recommendations'))

    # Plan cache bloat
    for r in records.get('plan_cache', []):
        plan_count = safe_val(r, 'plan_count', 0)
        single_use = safe_val(r, 'single_use_count', 0)
        single_mb = safe_val(r, 'single_use_size_mb', 0)
        if plan_count > 0:
            single_pct = (single_use/plan_count)*100
            if single_pct > config.plan_cache_single_use_pct and single_mb > 50:
                recs.append(make_rec('OPERATIONS', 'PLAN_CACHE_BLOAT', 'MEDIUM', 'DATABASE',
                    '', f'Plan Type: {r.get("plan_type","")}',
                    'Plan cache bloated with single-use plans',
                    f'{single_pct:.0f}% ({single_use:,}) are single-use, consuming {single_mb:.1f} MB. Consider parameterization.',
                    estimated_savings_mb=single_mb, risk='MEDIUM', confidence='MEDIUM',
                    source_dmv='sys.dm_exec_cached_plans'))

    # Log space
    for r in records.get('log_space', []):
        log_pct = safe_val(r, 'used_log_space_in_percent', 0)
        if log_pct > 80:
            sev = 'HIGH' if log_pct > 90 else 'MEDIUM'
            recs.append(make_rec('OPERATIONS', 'LOG_SPACE', sev, 'DATABASE',
                '', config.database,
                'Transaction log is nearly full',
                f'Log usage: {log_pct:.1f}% ({safe_val(r,"used_log_space_mb"):.1f} MB of {safe_val(r,"total_log_size_mb"):.1f} MB)',
                estimated_impact_pct=20.0, effort='LOW', risk='MEDIUM',
                source_dmv='sys.dm_db_log_space_usage'))

    return recs

ops_recs = analyze_operations(records, config)
print(f'Operations Recommendations: {len(ops_recs)}')

## 🏆 Score, Rank & Build Executive Summary

In [ ]:
from itertools import chain as iterchain
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Combine ALL 10 analyzers
all_recs = (index_recs + sp_recs + view_recs + schema_recs + storage_recs +
            activity_recs + archival_recs + perf_recs + cost_recs + ops_recs)
print(f'Total recommendations: {len(all_recs)}')

# Build Spark DataFrame
rec_schema = StructType([
    StructField('category', StringType()), StructField('subcategory', StringType()),
    StructField('severity', StringType()), StructField('object_type', StringType()),
    StructField('schema_name', StringType()), StructField('object_name', StringType()),
    StructField('recommendation', StringType()), StructField('recommendation_detail', StringType()),
    StructField('action_sql', StringType()),
    StructField('estimated_impact_pct', DoubleType()), StructField('estimated_savings_mb', DoubleType()),
    StructField('estimated_cost_monthly', DoubleType()),
    StructField('effort', StringType()), StructField('risk', StringType()),
    StructField('confidence', StringType()), StructField('source_dmv', StringType()),
    StructField('metric_value', StringType()), StructField('metric_threshold', StringType()),
])

if all_recs:
    result = spark.createDataFrame(all_recs, schema=rec_schema)
else:
    result = spark.createDataFrame([], schema=rec_schema)

result = (result
    .withColumn('analysis_date', F.current_date())
    .withColumn('analysis_timestamp', F.current_timestamp())
    .withColumn('database_name', F.lit(config.database)))

# Severity scoring
_sev_map = F.create_map(*iterchain.from_iterable(
    (F.lit(k), F.lit(v)) for k, v in SEVERITY_SCORES_STR.items()))
_cat_wt_map = F.create_map(*iterchain.from_iterable(
    (F.lit(k), F.lit(getattr(config, v))) for k, v in CATEGORY_WEIGHT_FIELDS.items()))

result = (result
    .withColumn('base_score', F.coalesce(_sev_map[F.col('severity')], F.lit(25)))
    .withColumn('category_weight', F.coalesce(_cat_wt_map[F.col('category')], F.lit(0.10)))
    .withColumn('priority_score',
        F.least(F.lit(100),
            (F.col('base_score') + F.col('estimated_impact_pct') * 0.3) * (F.lit(0.5) + F.col('category_weight'))))
    .withColumn('priority_rank', F.dense_rank().over(Window.orderBy(F.col('priority_score').desc()))))

# Health score (100 - weighted penalty)
total = result.count()
crit = result.filter(F.col('severity')=='CRITICAL').count()
high = result.filter(F.col('severity')=='HIGH').count()
med  = result.filter(F.col('severity')=='MEDIUM').count()
health_score = max(0, 100 - crit*15 - high*5 - med*1)

print(f'\n{"="*60}')
print(f'  HEALTH SCORE: {health_score}/100')
print(f'  Total: {total} | Critical: {crit} | High: {high} | Medium: {med}')
print(f'{"="*60}')

# Category summary
cat_seed = spark.createDataFrame([(c,) for c in ANALYSIS_CATEGORIES], StructType([StructField('category', StringType())]))
cat_summary = (cat_seed.join(
    result.groupBy('category').agg(
        F.count(F.lit(1)).alias('count'),
        F.sum(F.when(F.col('severity')=='CRITICAL',1).otherwise(0)).alias('critical'),
        F.sum(F.when(F.col('severity')=='HIGH',1).otherwise(0)).alias('high'),
        F.round(F.sum(F.coalesce(F.col('estimated_savings_mb'), F.lit(0.0)))/1024.0, 2).alias('savings_gb'),
    ), on='category', how='left')
    .fillna({'count':0, 'critical':0, 'high':0, 'savings_gb':0.0})
    .orderBy('category'))

display(cat_summary)

## 🗄️ Persist Recommendations to ADLS Gen2 (Parquet)

In [ ]:
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

output_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/recommendations/{server_name}/{database_name}"
now = datetime.now(timezone.utc)
partition = f"year={now.year}/month={now.month:02d}/day={now.day:02d}"

# Select final columns
final_cols = [
    'analysis_date', 'analysis_timestamp', 'database_name',
    'category', 'subcategory', 'severity', 'priority_score', 'priority_rank',
    'object_type', 'schema_name', 'object_name',
    'recommendation', 'recommendation_detail', 'action_sql',
    'estimated_impact_pct', 'estimated_savings_mb', 'estimated_cost_monthly',
    'effort', 'risk', 'confidence', 'source_dmv',
    'metric_value', 'metric_threshold',
]

output_df = result.select(*[F.col(c) for c in final_cols if c in result.columns])

# Write partitioned Parquet
output_df.write.mode('overwrite').parquet(f'{output_path}/{partition}')
print(f'✅ Recommendations saved: {output_path}/{partition}')
print(f'   Total: {output_df.count()} recommendations')

## 📄 Generate HTML Report

In [ ]:
def generate_html_report(ranked_recs, health_score, category_summary_rows, db_name):
    sev_colors = {'CRITICAL':'#dc2626','HIGH':'#ea580c','MEDIUM':'#ca8a04','LOW':'#16a34a','INFO':'#6b7280'}
    html = f'''<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Azure SQL Advisor — {db_name}</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; margin: 2em; background: #f8fafc; color: #1e293b; }}
h1 {{ color: #0f172a; }} h2 {{ color: #334155; border-bottom: 2px solid #e2e8f0; padding-bottom: 0.5em; }}
.score {{ font-size: 3em; font-weight: bold; color: {"#16a34a" if health_score >= 70 else ("#ca8a04" if health_score >= 40 else "#dc2626")}; }}
table {{ border-collapse: collapse; width: 100%; margin: 1em 0; }}
th {{ background: #1e293b; color: white; padding: 10px; text-align: left; }}
td {{ padding: 8px; border-bottom: 1px solid #e2e8f0; }}
tr:hover {{ background: #f1f5f9; }}
.pill {{ padding: 2px 8px; border-radius: 4px; color: white; font-size: 0.85em; font-weight: bold; }}
</style></head><body>
<h1>🚀 Azure SQL Advisor Report — {db_name}</h1>
<p>Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}</p>
<h2>Health Score</h2><div class="score">{health_score}/100</div>
<h2>Category Summary</h2><table><tr><th>Category</th><th>Count</th><th>Critical</th><th>High</th></tr>'''
    for row in category_summary_rows:
        html += f'<tr><td>{row["category"]}</td><td>{row.get("count",0)}</td><td>{row.get("critical",0)}</td><td>{row.get("high",0)}</td></tr>'
    html += '</table>'
    html += '<h2>Top Recommendations</h2><table><tr><th>#</th><th>Severity</th><th>Category</th><th>Object</th><th>Recommendation</th><th>Impact</th></tr>'
    for i, rec in enumerate(ranked_recs[:50], 1):
        sev = rec.get('severity', 'INFO')
        color = sev_colors.get(sev, '#6b7280')
        html += f'<tr><td>{i}</td><td><span class="pill" style="background:{color}">{sev}</span></td><td>{rec.get("category","")}</td><td>{rec.get("object_name","")}</td><td>{rec.get("recommendation","")}<br><small>{rec.get("recommendation_detail","")[:200]}</small></td><td>{rec.get("estimated_impact_pct",0):.0f}%</td></tr>'
    html += '</table></body></html>'
    return html

# Generate report
ranked_rows = [row.asDict() for row in result.orderBy(F.col('priority_rank').asc()).collect()]
cat_rows = [row.asDict() for row in cat_summary.collect()]
html_report = generate_html_report(ranked_rows, health_score, cat_rows, database_name)

# Save HTML to ADLS Gen2
report_path = f'{output_path}/reports/{database_name}_report_{now.strftime("%Y%m%d_%H%M%S")}.html'
dbutils.fs.put(report_path.replace('abfss://', 'dbfs:/').replace('.dfs.core.windows.net', ''), html_report, overwrite=True)
print(f'✅ HTML report saved: {report_path}')

In [ ]:
# Notebook exit value for orchestration
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "health_score": health_score,
    "total_recommendations": len(all_recs),
    "critical_count": crit,
    "high_count": high,
    "categories": len(ANALYSIS_CATEGORIES),
    "output_path": output_path,
    "timestamp": now.isoformat(),
}))